# Embedding experiments

## Imports

In [38]:
# Dataset management
import os
import numpy as np
import pandas as pd

# Embeddings
from sentence_transformers import SentenceTransformer, util

## Load data

In [39]:
RAW_CSV = "../data/raw/booksummaries.csv"
CLEAN_CSV = "../data/processed/processed_booksummaries.csv"

if os.path.exists(CLEAN_CSV) and os.path.exists(RAW_CSV):
    df = pd.read_csv(CLEAN_CSV)
    raw = pd.read_csv(RAW_CSV, usecols=["wikipedia_id", "synopsis"]).rename(columns={"synopsis": "synopsis_raw"})
else:
    raise FileNotFoundError(f"Dataset not found at {CLEAN_CSV} or {RAW_CSV}")

df = df.merge(raw, on="wikipedia_id", how="left", validate="many_to_one")

## Generate Text Embeddings

In [40]:
# Fill missing values with empty strings to prevent NaN results
df["combined text"] = (
    "Title: " + df["title"].fillna("") + " | " +
    "Author: " + df["author"].fillna("Unknown Author") + " | " +
    "Genres: " + df["genres"].fillna("Unknown Genre") + " | " +
    "Synopsis: " + df["synopsis"].fillna("")
)

In [41]:
model = SentenceTransformer("all-MiniLM-L6-v2")
book_embeddings = model.encode(
    df["combined text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

In [42]:
EMBEDDINGS_NPY = "../data/embeddings/booksummaries.npy"

np.save(EMBEDDINGS_NPY, book_embeddings.astype(np.float32))

## Try queries

In [43]:
queries = [
    "I want a dark psychological novel about guilt and morality",
    "An emotional science-fiction story exploring human relationships and artificial intelligence",
    "A classic romantic story with strong female characters and social criticism",
    "Epic fantasy with dragons, quests and friendship",
    "A quiet philosophical book about loneliness and the meaning of life"
]

In [44]:
def find_most_similar(query: str, top_k: int = 5):
    """Return the top-k most similar books to a free-text query."""
    query_emb = model.encode(query, normalize_embeddings=True)
    scores = util.cos_sim(query_emb, book_embeddings)[0].cpu().numpy()
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "rank": len(results) + 1,
            "title": df.iloc[idx]["title"],
            "author": df.iloc[idx]["author"],
            "genre": df.iloc[idx]["genres"],
            "score": float(scores[idx]),
            "synopsis_preview": df.iloc[idx]["synopsis_raw"][:180] + "..."
        })
    return pd.DataFrame(results)

In [45]:
for q in queries:
    print("\n" + "="*80)
    print(f"QUERY: {q}")
    print("="*80)
    display(find_most_similar(q, top_k=5))


QUERY: I want a dark psychological novel about guilt and morality


,rank,title,author,genre,score,synopsis_preview
0,1,A Mind to Murder,P. D. James,"Crime Fiction, Mystery",0.527390,"In a psychiatric clinic late one night, the p..."
1,2,Disordered Minds,Minette Walters,"Crime Fiction, Mystery",0.527269,"In 1970 Harold Stump, a mentally challenged y..."
2,3,The Darkling,David Kesterton,"Fantasy, Speculative fiction",0.522443,"Set in the distant future, a young tribesman,..."
3,4,Dark Reflections,Samuel R. Delany,"Fiction, Novel, Speculative fiction",0.521831,Dark Reflections tells the story of Arnold Ha...
4,5,The Torment of Others,Val McDermid,"Crime Fiction, Fiction, Mystery, Suspense",0.512172,Several years after Derek Tyler was incarcera...



QUERY: An emotional science-fiction story exploring human relationships and artificial intelligence


,rank,title,author,genre,score,synopsis_preview
0,1,Headlong,Simon Ings,Science Fiction,0.496797,Headlong is set England in the mid-21st centu...
1,2,Eager,Helen Fox,"Children's literature, Science Fiction",0.464427,The plot is set in England at the end of the ...
2,3,The Positronic Man,Isaac Asimov,"Fiction, Science Fiction",0.428826,In the twenty-first century the creation of t...
3,4,Mathematicians in Love,Rudy Rucker,"Science Fiction, Speculative fiction",0.412400,Bela and Paul are working towards their Ph.D'...
4,5,Death Match: A Novel,Lincoln Child,"Horror, Science Fiction, Speculative fiction, ...",0.403445,"Every once in a rare while, the most perfect ..."



QUERY: A classic romantic story with strong female characters and social criticism


,rank,title,author,genre,score,synopsis_preview
0,1,The Romantics,Pankaj Mishra,Literary fiction,0.557132,"Samar, the young narrator of The Romantics, a..."
1,2,La pensée straight,Monique Wittig,Essay,0.550742,"""One Is Not Born a Woman,"" follows in the wak..."
2,3,Pubis angelical,Manuel Puig,"Fiction, Novel",0.544630,The narrative alternates between separate nar...
3,4,Morality for Beautiful Girls,Alexander McCall Smith,"Detective fiction, Mystery",0.522783,"Mma Ramotswe is engaged to ""the excellent"" Mr..."
4,5,Hideous Kinky,Esther Freud,"Autobiography, Fiction, Novel",0.506811,A young mother and her two daughters travel t...



QUERY: Epic fantasy with dragons, quests and friendship


,rank,title,author,genre,score,synopsis_preview
0,1,Dragons of Summer Flame,Margaret Weis,"Fantasy, Fiction, Speculative fiction",0.592921,"After a battle, the Nightlord Steel Brightbla..."
1,2,Dragonquest,Anne McCaffrey,"Children's literature, Fantasy, Fiction, Scien...",0.592703,"As it opens, tensions are rising between the ..."
2,3,A Dragon at Worlds' End,Christopher Rowley,"Fantasy, Speculative fiction",0.590521,Trapped and alone on the dark continent of Ei...
3,4,Dragons of Winter Night,Tracy Hickman,"Children's literature, Fantasy, Fiction, Specu...",0.588273,The novel begins with the Companions assemble...
4,5,Dragonflame,Graham Edwards,"Fantasy, Speculative fiction",0.583744,Aria joins the sinister Cult of the Last Circ...



QUERY: A quiet philosophical book about loneliness and the meaning of life


,rank,title,author,genre,score,synopsis_preview
0,1,The Well of Loneliness,Radclyffe Hall,"Künstlerroman, Novel",0.510821,"The book's protagonist, Stephen Gordon, is bo..."
1,2,All Alone,Claire Huchet Bishop,Children's literature,0.469752,"Marcel Mabout is a ""ten-year-old man"", sent b..."
2,3,The New Life,Orhan Pamuk,"Fiction, Novel",0.464735,The plot centers around a young engineering s...
3,4,In a Lonely Place,Dorothy B. Hughes,Mystery,0.462588,The novel is a noir set in post World War II ...
4,5,A Single Man,Christopher Isherwood,Gay novel,0.453904,"An English professor, about a year after the ..."
